# MCP Deep Dive — Instructor Walkthrough

This notebook is the live-lecture companion for our session on the **Model Context Protocol (MCP)**. We will build intuition for MCP by inspecting servers, clients, tools, resources, and prompts in-process — then point at the CLI starter project where students will assemble the full chatbot.

## Agenda

1. Introducing MCP
2. MCP Clients
3. Project Setup
4. Defining Tools with MCP
5. The Server Inspector
6. Implementing a Client
7. Defining Resources
8. Accessing Resources
9. Defining Prompts
10. Prompts in the Client

We close with a recap and a short exercise list students can pick up after class.

## Setup

Before running anything in this notebook:

1. Create a `.env` file in the project root with your key: `ANTHROPIC_API_KEY=sk-ant-...`
2. Add `.env` to your `.gitignore` so the key never lands in version control.
3. Install the SDKs we will use throughout the lecture (next cell — uncomment the line that fits your environment).

We rely on three packages today:

| Package | Purpose |
| --- | --- |
| `anthropic` | Talk to Claude from Python |
| `python-dotenv` | Load the API key from `.env` |
| `mcp[cli]` | Build MCP servers and clients **plus** the `mcp dev` inspector launcher (the `[cli]` extras pull in `typer`) |

In [ ]:
# %pip install anthropic python-dotenv 'mcp[cli]'
# or, with uv:
# !uv pip install anthropic python-dotenv 'mcp[cli]'
#
# Note: the [cli] extras pull in typer + the `mcp dev` inspector launcher.
# Plain `mcp` alone is enough for code-only use, but the section 5 demo
# (`mcp dev mcp_server.py`) will fail with "typer is required" without it.
# In zsh, the single quotes around 'mcp[cli]' are required so the shell
# doesn't try to glob the brackets.

### Initialize the Anthropic client

The next cell loads the API key from `.env`, creates the Anthropic client, and prints a three-line sanity check: SDK version, the model we will default to, and whether the key actually loaded. If you see `Key loaded: False`, stop here — every later cell will fail until the `.env` file is in place.

In [ ]:
import os
import anthropic
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
model = "claude-sonnet-4-6"

print(f"anthropic SDK version: {anthropic.__version__}")
print(f"Default model: {model}")
print(f"Key loaded: {bool(os.getenv('ANTHROPIC_API_KEY'))}")

### Shared helpers

We define a small `chat()` helper plus message builders once, here. Every later demo that calls Claude reuses these — when a section's code is short, students can focus on the *one* thing that varies (a system prompt, a tool result, a prompt template) instead of the boilerplate.

In [ ]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})
    return messages

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})
    return messages

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    """Send messages to Claude and return the assistant text."""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system is not None:
        params["system"] = system
    if stop_sequences is not None:
        params["stop_sequences"] = stop_sequences
    response = client.messages.create(**params)
    return response.content[0].text

---
# 1. Introducing MCP

**MCP (Model Context Protocol)** is a communication layer that gives Claude access to tools, resources, and prompts — without forcing application developers to author every schema and integration function by hand.

### The architecture, at a glance

```
  Your application server
        │
        │  (uses)
        ▼
  ┌─────────────┐         ┌─────────────────────────┐
  │  MCP Client │ ──────▶ │       MCP Server        │
  └─────────────┘         │  ┌────┐ ┌───────┐ ┌───┐ │
                          │  │Tools│ │Resources│ │Prompts│
                          │  └────┘ └───────┘ └───┘ │
                          └─────────────────────────┘
                                     │
                                     ▼
                              External service
                              (GitHub, AWS, DB...)
```

### What problem does it actually solve?

Imagine you are building a GitHub-aware chatbot. Without MCP, *you* must write tool schemas and Python implementations for repositories, pull requests, issues, projects, comments, reviews — pages of repetitive code that you also need to keep in sync with GitHub's API.

With MCP, GitHub (or a community maintainer) ships an MCP server that already exposes those tools. Your application connects an MCP client and gets the tool list for free.

### Common questions

| Question | Short answer |
| --- | --- |
| Who creates MCP servers? | Anyone — but service providers (AWS, GitHub, etc.) often ship official ones. |
| How is this different from calling the API directly? | You stop authoring tool schemas and execution functions yourself. |
| How does this relate to tool use? | Complementary — tool use still happens, but the *server* defines and runs the tools instead of you. |

**Core value:** MCP shifts the integration burden from application developers to MCP server maintainers.

### Demo: hand-rolled tool schema vs. MCP decorator

This cell prints a side-by-side comparison: the verbose JSON schema you would normally write for a single `read_doc_contents` tool, then the much shorter MCP equivalent. Watch how much repetition disappears — and notice that the MCP version is the source of truth for *both* the schema and the implementation.

In [ ]:
import json

manual_schema = {
    "name": "read_doc_contents",
    "description": "Read the contents of a document by its ID.",
    "input_schema": {
        "type": "object",
        "properties": {
            "doc_id": {
                "type": "string",
                "description": "The ID of the document to read",
            }
        },
        "required": ["doc_id"],
    },
}

mcp_equivalent = '''
@mcp.tool()
def read_doc_contents(
    doc_id: str = Field(description="The ID of the document to read"),
) -> str:
    """Read the contents of a document by its ID."""
    return docs[doc_id]
'''

print("=== Hand-written tool definition ===")
print(json.dumps(manual_schema, indent=2))
print("\n=== MCP equivalent (schema auto-generated) ===")
print(mcp_equivalent)

> **🏫 During class:**
> 1. Run the demo cell, then put the two outputs side by side on screen.
> 2. Say: *"Both define the exact same tool — but the MCP version also tells you how to **run** it. Two responsibilities, one decorator."*
> 3. Ask the room: *"How many tools would the GitHub MCP server need? Show of hands — over twenty?"* (It is — the official server has 50+.)

---
# 2. MCP Clients

An **MCP Client** is the communication interface between your application server and an MCP server. Your code never speaks to the MCP server directly — it goes through the client.

### Transport agnostic

Clients and servers can communicate over multiple protocols:

| Transport | When you'd use it |
| --- | --- |
| `stdio` | Same machine, local process — the common case for development |
| `HTTP` (Streamable / SSE) | Remote MCP server hosted as a service |
| `WebSockets` | Less common, full-duplex remote scenarios |

### The four message types you must remember

| Direction | Message | Meaning |
| --- | --- | --- |
| client → server | `list_tools` request | "What tools do you have?" |
| server → client | `list_tools` result | "Here is the catalog." |
| client → server | `call_tool` request | "Run this tool with these args." |
| server → client | `call_tool` result | "Here is the output." |

### The full flow (10 steps)

1. User sends a query to your application server
2. Your server asks the MCP client for available tools
3. MCP client sends a `list_tools` request to the MCP server
4. MCP server returns a `list_tools` result
5. Your server sends `query + tools` to Claude
6. Claude requests a tool execution
7. Your server tells the MCP client to run the tool
8. MCP client sends a `call_tool` request to the MCP server
9. MCP server actually executes the tool (e.g. hits the GitHub API)
10. Result flows back: MCP server → MCP client → your server → Claude → user

### Demo: simulate the message flow

Rather than wiring up a real network, this cell prints the same 10-step exchange as labeled message dictionaries. The point is to show students that MCP messages are just JSON-shaped envelopes — nothing magical, nothing async-looking on paper.

In [ ]:
exchange = [
    ("user → server", "Tell me the contents of doc 'plan.md'"),
    ("server → client", "Need tools, please."),
    ("client → MCP server", {"method": "tools/list"}),
    ("MCP server → client", {"tools": ["read_doc_contents", "edit_document"]}),
    ("server → Claude", "messages + tool catalog"),
    ("Claude → server", {"tool_use": "read_doc_contents", "input": {"doc_id": "plan.md"}}),
    ("server → client", "Please call read_doc_contents"),
    ("client → MCP server", {"method": "tools/call", "name": "read_doc_contents", "args": {"doc_id": "plan.md"}}),
    ("MCP server → external", "open file or hit API"),
    ("MCP server → client → server → Claude → user", "contents of plan.md"),
]

for i, (hop, payload) in enumerate(exchange, start=1):
    print(f"{i:>2}. {hop}")
    print(f"     {payload}\n")

> **🏫 During class:**
> 1. Run the cell and walk down the printed list line by line.
> 2. Say: *"Notice that Claude never talks to the MCP server. The application server is the broker — Claude only sees tools and tool results."*
> 3. Ask: *"Which step would change if we hosted the MCP server remotely instead of as a subprocess?"* (Only step 3 and step 8 — the transport — everything else is identical.)

---
# 3. Project Setup

We will reference a small **CLI-based chatbot project** throughout the rest of the lecture. Students will receive a starter zip and progressively wire up MCP features.

### What's in the starter

| Component | Role |
| --- | --- |
| `MCP client` | Connects to our custom MCP server |
| `MCP server` | Exposes 2 tools: read document, update document |
| `docs` dictionary | A few fake documents kept entirely in memory |

> ℹ️ Real projects implement *either* a client *or* a server, almost never both. We do both here so students see the full loop end-to-end.

### Setup steps for students

1. Download `CLI_project.zip` from the course materials
2. Extract and open in your editor
3. Follow `readme.md` for environment setup
4. Add `ANTHROPIC_API_KEY` to `.env`
5. Install dependencies (with or without `uv`)
6. Run: `uv run main.py` *(or)* `python main.py`
7. Send a chat prompt — you should get a response

If step 7 works, you have a baseline chatbot. The rest of the lecture adds MCP on top.

### Demo: verify your environment is ready

This cell imports `mcp`, `anthropic`, and `dotenv`, then prints versions plus whether the API key is loaded. If any import fails, the corresponding install step in the previous section was skipped. Run this *before* class so you catch broken environments early.

In [ ]:
import sys
import importlib.metadata as md

for pkg in ["anthropic", "mcp", "python-dotenv"]:
    try:
        version = md.version(pkg)
        print(f"  ok   {pkg:<14} {version}")
    except md.PackageNotFoundError:
        print(f"  MISS {pkg:<14} (run pip install)")

print(f"\nPython: {sys.version.split()[0]}")
print(f"Key loaded: {bool(os.getenv('ANTHROPIC_API_KEY'))}")

> **🏫 During class:**
> 1. Run this cell first — if anything is `MISS`, fix the install before moving on.
> 2. Say: *"Two SDKs and a dotenv loader. That is everything we need to build both halves of an MCP integration."*
> 3. Ask: *"Why does the starter project keep documents in memory instead of using a database?"* (To keep the focus on MCP mechanics, not persistence.)

---
# 4. Defining Tools with MCP

The MCP Python SDK lets you define tools with a Python decorator instead of writing JSON schemas. The SDK reads your function's type annotations and `Field()` descriptions to generate the JSON schema automatically.

### The pattern

```python
from mcp.server.fastmcp import FastMCP
from pydantic import Field

mcp = FastMCP("DocumentMCP")

@mcp.tool()
def my_tool(
    arg1: str = Field(description="What arg1 means"),
) -> str:
    """Short description of what this tool does."""
    ...
```

### What the SDK gives you

| Inferred from... | ...maps to... |
| --- | --- |
| `@mcp.tool()` decorator | The tool exists and is exposed |
| Function name | The tool's `name` |
| Docstring | The tool's `description` |
| Parameter types | JSON schema types in `inputSchema` |
| `Field(description=...)` | Per-argument descriptions |
| Required vs default arguments | `required` array in the schema |

### Error handling

Raise `ValueError` (or any exception) inside the function — MCP returns it as a tool error to the client, which forwards it to Claude. No special wrapping needed.

### Demo: define two tools and inspect the auto-generated schemas

We build the same two tools the starter project uses (`read_doc_contents`, `edit_document`), then call `mcp.list_tools()` to print the JSON schemas the SDK produced. Watch how the `Field(description=...)` text shows up directly in the per-argument descriptions — that string is what Claude sees when deciding which tool to call.

In [ ]:
from mcp.server.fastmcp import FastMCP
from pydantic import Field

docs = {
    "plan.md": "# Q2 Plan\nShip MCP support and write the lecture.",
    "notes.md": "- ask about budget\n- ping legal",
    "readme.md": "This project demonstrates MCP for the workshop.",
}

mcp_server = FastMCP("DocumentMCP")

@mcp_server.tool()
def read_doc_contents(
    doc_id: str = Field(description="The ID of the document to read"),
) -> str:
    """Read the contents of a document by its ID."""
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    return docs[doc_id]

@mcp_server.tool()
def edit_document(
    doc_id: str = Field(description="The ID of the document to edit"),
    old_string: str = Field(description="Exact text to replace"),
    new_string: str = Field(description="Replacement text"),
) -> str:
    """Replace `old_string` with `new_string` inside the named document."""
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    docs[doc_id] = docs[doc_id].replace(old_string, new_string)
    return docs[doc_id]

tools = await mcp_server.list_tools()
for tool in tools:
    print(f"--- {tool.name} ---")
    print(f"description: {tool.description}")
    print("inputSchema:")
    print(json.dumps(tool.inputSchema, indent=2))
    print()

> **🏫 During class:**
> 1. Run the cell and pull up one of the printed `inputSchema` blocks.
> 2. Say: *"Every line in this schema came from a Python annotation. Change the `Field(description=...)` text and the schema updates next run — your code stays single-source."*
> 3. Live edit: change `read_doc_contents`'s docstring to add the word `Markdown` and re-run. Show that the `description` field updates without touching any JSON.

---
# 5. The Server Inspector

The **MCP Inspector** is an in-browser debugger for your MCP server. It connects to a server, lets you click any tool/resource/prompt, fill in parameters via a form, and see the raw response — without writing any client code.

### How to launch it

From the project directory, run:

```bash
mcp dev path/to/server.py
```

The CLI prints a local URL (typically `http://localhost:5173/?...`). Open it in your browser.

### What the UI gives you

| Region | Purpose |
| --- | --- |
| Left sidebar | A `Connect` button to bind to your running server |
| Top tabs | Sections for **Resources**, **Prompts**, **Tools** |
| Tool list | Each tool clickable; opens an input form |
| Right panel | Form + live result of every invocation |

> ⚠️ The UI changes regularly. Tabs and labels shift between releases — the workflow (connect → pick component → invoke → read result) does not.

### Why use it

- Smoke-test new tools the moment you write them — no chatbot needed.
- Diff the schema view against what you *expected* to expose.
- Chain invocations to verify state changes (e.g. `edit_document` then `read_doc_contents`).

### Demo: print the launch command and what students should see

There is no notebook UI for the inspector — it lives in the browser. Instead, this cell prints the exact command students should run from the project root, plus the manual checklist they should walk through once the page loads.

In [ ]:
command = "mcp dev mcp_server.py"

checklist = [
    "Open the printed URL in your browser",
    "Click Connect in the left sidebar",
    "Switch to the Tools tab",
    "Click read_doc_contents — in the doc_id field type just: plan.md (no quotes, no 'doc_id=' prefix)",
    "Click edit_document — fill the three fields with values only (e.g. doc_id: plan.md, old_string: Q2, new_string: Q3)",
    "Re-run read_doc_contents to confirm the change persisted",
]

print(f"Run this in your project directory:\n  $ {command}\n")
print("Then in the browser:")
for i, step in enumerate(checklist, 1):
    print(f"  {i}. {step}")

> **🏫 During class:**
> 1. Open a terminal alongside the notebook and actually run `mcp dev` against your demo server.
> 2. Say: *"This is the fastest feedback loop you'll get when authoring an MCP server. If a tool misbehaves here, it'll misbehave in your chatbot too."*
> 3. Variation: deliberately call `read_doc_contents` with a missing doc id — show students the inspector surfaces the `ValueError` cleanly.

---
# 6. Implementing a Client

On the application side, you wrap an MCP **`ClientSession`** in a small class so you can manage the connection lifecycle and expose ergonomic methods to the rest of your codebase.

### Why wrap the session?

- The raw `ClientSession` requires resource cleanup on close.
- Application code shouldn't care whether tools come from MCP or somewhere else — it just wants `list_tools()` and `call_tool(name, args)`.
- A wrapper makes it easy to swap transports (stdio ↔ HTTP) later.

### The two methods you actually need

```python
class MCPClient:
    async def list_tools(self):
        result = await self.session.list_tools()
        return result.tools

    async def call_tool(self, tool_name, tool_input):
        return await self.session.call_tool(tool_name, tool_input)
```

Everything else — connection setup, transport configuration, retries — lives inside the wrapper class. The rest of your application server only ever calls these two methods (plus the resource and prompt methods we add in §8 and §10).

### Demo: connect a client to our server in-process and call a tool

We use `create_connected_server_and_client_session` from `mcp.shared.memory` to wire the server we built in §4 to a fresh client *without spinning up a subprocess*. This is the same harness MCP's own tests use — perfect for notebooks. Watch the printed `tool.name` list, then the actual tool result returned by `call_tool`.

In [ ]:
from mcp.shared.memory import create_connected_server_and_client_session

async with create_connected_server_and_client_session(
    mcp_server._mcp_server
) as session:
    list_result = await session.list_tools()
    print("Tools available to the client:")
    for t in list_result.tools:
        print(f"  - {t.name}")

    print("\nCalling read_doc_contents(doc_id='plan.md')...")
    call_result = await session.call_tool(
        "read_doc_contents", {"doc_id": "plan.md"}
    )
    print("Result content blocks:")
    for block in call_result.content:
        print(f"  [{block.type}] {block.text}")

> **🏫 During class:**
> 1. Run the cell and point at the `Tools available` list — note the names match what we registered in §4.
> 2. Say: *"Your application server only ever talks to this `session` object. It does not know whether the server lives in-process, in a subprocess, or behind an HTTPS endpoint."*
> 3. Variation: call `edit_document` to change one word in `plan.md`, then re-run `read_doc_contents` and confirm the in-memory state mutated.

---
# 7. Defining Resources

**Resources** let an MCP server expose *data* the client can read on demand — distinct from tools, which expose *actions* the model can invoke.

### Two flavors

| Type | URI shape | Use it for |
| --- | --- | --- |
| Direct | `docs://documents` | A single fixed payload (e.g. "the catalog") |
| Templated | `docs://documents/{doc_id}` | Parameterized lookups; SDK parses `{doc_id}` and passes it as a kwarg |

### URI = the address

Each resource carries a URI you choose at definition time. Clients fetch by URI; the server matches the URI to the right Python function and runs it.

### MIME types

The MIME type is a hint to the client about how to parse the response:

- `application/json` → client should `json.loads` the text
- `text/plain` → return the text as-is

### Resources vs. Tools — the mental model

| | Resources | Tools |
| --- | --- | --- |
| Triggered by | Client decision (e.g. `@mention`) | Claude's decision |
| Side effects | Read-only by convention | Anything |
| When loaded | Proactively, before chatting | Reactively, mid-turn |

If the data should *always* be in context (e.g. a document the user just attached), make it a resource. If the model should *decide* when to fetch, make it a tool.

### Demo: add a direct + a templated resource and inspect them

We register one direct resource (`docs://documents` returns the list of doc IDs as JSON) and one templated resource (`docs://documents/{doc_id}` returns a single doc's text). Then we list both via the client. Watch the `mimeType` field — it differs by resource and tells the client how to parse later.

In [ ]:
@mcp_server.resource("docs://documents", mime_type="application/json")
def list_docs() -> list[str]:
    """Return all available document IDs."""
    return list(docs.keys())

@mcp_server.resource("docs://documents/{doc_id}", mime_type="text/plain")
def fetch_doc(doc_id: str) -> str:
    """Return the full text of a single document."""
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    return docs[doc_id]

async with create_connected_server_and_client_session(
    mcp_server._mcp_server
) as session:
    direct = await session.list_resources()
    print("Direct resources:")
    for r in direct.resources:
        print(f"  - {r.uri} ({r.mimeType})")

    templated = await session.list_resource_templates()
    print("\nTemplated resources:")
    for r in templated.resourceTemplates:
        print(f"  - {r.uriTemplate} ({r.mimeType})")

> **🏫 During class:**
> 1. Run the cell and highlight that direct vs. templated resources show up in *separate* listing calls.
> 2. Say: *"Templated resources are basically URL patterns. The SDK parses `{doc_id}` for you and hands it to your function as a keyword argument."*
> 3. Ask: *"Should `read_doc_contents` be a tool or a resource?"* (Either works — but if the user just typed `@plan.md`, fetching as a resource keeps it out of Claude's tool budget.)

---
# 8. Accessing Resources

On the client side, you add a third method to your wrapper that fetches a resource by URI and returns the *deserialized* payload.

### The implementation

```python
from pydantic import AnyUrl
import json

async def read_resource(self, uri: str):
    result = await self.session.read_resource(AnyUrl(uri))
    resource = result.contents[0]
    if resource.mimeType == "application/json":
        return json.loads(resource.text)
    return resource.text
```

### What's happening line-by-line

| Line | Why it matters |
| --- | --- |
| `AnyUrl(uri)` | The session expects a Pydantic URL, not a plain `str` |
| `result.contents[0]` | The server can return multiple content blocks; we use the first |
| `resource.mimeType` | Drives the parsing strategy on the next line |
| `json.loads` vs `text` | The server already serialized to a string — we deserialize per MIME type |

### Where this gets used

Application code typically calls `read_resource` *before* sending the message to Claude — for example, when the user types `@plan.md`, the client fetches the doc and inlines it into the prompt. The model never sees a tool call for it; the content is just there.

### Demo: fetch both resource shapes and parse them correctly

We connect a client, hit the direct resource (gets back JSON list of doc IDs), then hit the templated resource (gets back the text of one doc). The output difference makes the MIME-type branching obvious — list of strings vs. raw markdown.

In [ ]:
from pydantic import AnyUrl

async def read_resource(session, uri: str):
    result = await session.read_resource(AnyUrl(uri))
    resource = result.contents[0]
    if resource.mimeType == "application/json":
        return json.loads(resource.text)
    return resource.text

async with create_connected_server_and_client_session(
    mcp_server._mcp_server
) as session:
    catalog = await read_resource(session, "docs://documents")
    print(f"Direct resource → Python type: {type(catalog).__name__}")
    print(f"  value: {catalog}\n")

    plan = await read_resource(session, "docs://documents/plan.md")
    print(f"Templated resource → Python type: {type(plan).__name__}")
    print(f"  value:\n{plan}")

> **🏫 During class:**
> 1. Run the cell and contrast the two `Python type` lines — one is a `list`, the other a `str`.
> 2. Say: *"The server already serialized everything to text. The client deserializes based on the MIME hint. That is the entire contract."*
> 3. Variation: try `await read_resource(session, "docs://documents/missing.md")` and let the room see the error bubble cleanly from the server.

---
# 9. Defining Prompts

**MCP Prompts** are pre-tested prompt templates the server hands to clients. Instead of every user inventing their own wording for a domain task, the server author ships a vetted prompt that does it well.

### Why this matters

If you maintain an MCP server, you understand its tools better than your users do. Prompts let you encode that expertise: "the right way to ask this server to format a document", "the right way to summarize an issue", and so on.

### Implementation

```python
from mcp.server.fastmcp.prompts import base

@mcp_server.prompt(
    name="format_document",
    description="Reformat a document into clean Markdown."
)
def format_doc_prompt(doc_id: str) -> list[base.Message]:
    return [
        base.UserMessage(
            f"Please read the document with id {doc_id}, "
            f"reformat it as clean Markdown, then save the changes "
            f"using the edit_document tool."
        )
    ]
```

### How clients see them

In a chat app, prompts typically surface as **slash commands** with autocomplete. The user types `/format_document`, the UI prompts for the `doc_id` argument, then dispatches the resulting messages straight to Claude.

### Demo: register a prompt and inspect it from the server side

We add a `format_document` prompt that takes a `doc_id`, then list prompts via `mcp_server.list_prompts()`. Pay attention to the `arguments` array — the SDK extracted `doc_id` from the function signature and surfaced it as a required argument the client must supply later.

In [ ]:
from mcp.server.fastmcp.prompts import base

@mcp_server.prompt(
    name="format_document",
    description="Reformat a document into clean Markdown.",
)
def format_doc_prompt(doc_id: str) -> list[base.Message]:
    return [
        base.UserMessage(
            f"Please read the document with id {doc_id}, "
            f"reformat it as clean Markdown, then save the changes "
            f"using the edit_document tool."
        )
    ]

prompts = await mcp_server.list_prompts()
for p in prompts:
    print(f"--- {p.name} ---")
    print(f"description: {p.description}")
    print("arguments:")
    for a in (p.arguments or []):
        print(f"  - {a.name} (required={a.required})")

> **🏫 During class:**
> 1. Run the cell and point at the printed `arguments` list — it came from the function signature.
> 2. Say: *"This is the same auto-introspection trick MCP uses for tools — the SDK reads your Python and turns it into protocol-shaped metadata."*
> 3. Question: *"Why ship prompts at all? Couldn't users write their own?"* (Yes — but the server author knows things users don't, like the exact tool order required to get good output.)

---
# 10. Prompts in the Client

On the client side, two methods complete the picture:

```python
async def list_prompts(self):
    result = await self.session.list_prompts()
    return result.prompts

async def get_prompt(self, prompt_name, arguments):
    result = await self.session.get_prompt(prompt_name, arguments)
    return result.messages
```

### The end-to-end flow

1. Server defines `@mcp_server.prompt(...)` with required arguments
2. Client calls `get_prompt(name, {arg: value})`
3. SDK passes the arguments as kwargs to your prompt function
4. Your function interpolates them into the prompt body
5. Returns a `messages` array — already shaped like Claude's `messages` parameter
6. Application code feeds those messages straight to Claude

### Why this is powerful

The prompt template is **versioned, tested, and shipped with the server**. When the server author improves the wording, every client gets the new version automatically — no chatbot redeploy needed.

### Demo: pull a prompt from the client and feed it to Claude

We list prompts, then call `get_prompt("format_document", {"doc_id": "plan.md"})`. The result is a `messages` array we can hand directly to our `chat()` helper. Watch how the `doc_id` argument flows from the client call into the rendered prompt text — and then how Claude's response references it back.

In [ ]:
async with create_connected_server_and_client_session(
    mcp_server._mcp_server
) as session:
    prompts_list = await session.list_prompts()
    print("Prompts the server advertises:")
    for p in prompts_list.prompts:
        print(f"  - {p.name}: {p.description}")

    prompt_result = await session.get_prompt(
        "format_document", {"doc_id": "plan.md"}
    )

    print("\nRendered messages (ready to send to Claude):")
    rendered = []
    for msg in prompt_result.messages:
        text = msg.content.text if hasattr(msg.content, "text") else str(msg.content)
        print(f"  [{msg.role}] {text}")
        rendered.append({"role": msg.role, "content": text})

print("\n--- Claude's response ---")
print(chat(rendered))

> **🏫 During class:**
> 1. Run the cell — point out that the `doc_id` argument we passed (`'plan.md'`) shows up *verbatim* in the rendered user message.
> 2. Say: *"This is the loop closing. Server defines wisdom, client invokes it with arguments, Claude executes against the same MCP tools we registered earlier."*
> 3. Variation: change the `doc_id` argument to `'notes.md'` and re-run. Same prompt template, different output — that is the value of templated prompts.

---
# Recap

- **MCP** is a protocol that splits responsibility: server authors define tools/resources/prompts; application authors connect a client and route between Claude and the server.
- **Tools** are actions the model can invoke; defined with `@mcp.tool()` — schemas auto-generate from type hints + `Field(description=...)`.
- **Resources** are read-only data exposed by URI; direct (`docs://documents`) or templated (`docs://documents/{doc_id}`); MIME type drives client parsing.
- **Prompts** are versioned templates that ship with the server, surfaced as slash commands in clients; arguments interpolate via function kwargs.
- **Clients** wrap `ClientSession` and expose four methods to the rest of your app: `list_tools`, `call_tool`, `read_resource`, `get_prompt`.
- **Inspector** (`mcp dev path/to/server.py`) is the fastest debug loop while authoring servers.

## Exercises

1. **Add a third tool** — `delete_document(doc_id)` — to the server. Verify it shows up via `mcp_server.list_tools()` *and* in the inspector.
2. **Add a `docs://documents/{doc_id}/metadata` resource** that returns `{"id": ..., "length": ...}` as JSON. Read it from the client and assert the parsed type is `dict`.
3. **Add a `summarize_document` prompt** taking `doc_id` and a `style` argument (`"bullets"` or `"paragraph"`). Render it both ways and compare Claude's output.
4. **Build a thin `MCPClient` class** that wraps `create_connected_server_and_client_session` as an async context manager and exposes the four canonical methods.
5. **Wire it into the CLI starter** so a real chatbot can call your server end-to-end.

In [ ]:
# Exercise 1 scaffold — finish the implementation, then run mcp_server.list_tools() to confirm.
#
# @mcp_server.tool()
# def delete_document(
#     doc_id: str = Field(description="The ID of the document to delete"),
# ) -> str:
#     """Delete the named document and return a confirmation string."""
#     ...
#
# tools = await mcp_server.list_tools()
# print([t.name for t in tools])